# 3D SOFFT Transformation Analysis

Interactive notebook for analyzing SO(3) registration results. Use the sliders to browse rotation candidates and volume slices.

In [2]:
# --- CONFIG ---
which_rotation = 0              # which rotation candidate (for all_solutions mode)
use_rotation_suffix = True      # True = 'Rotated0.csv' (all_solutions), False = 'Rotated.csv' (one_solution)

# --- DATA DIR ---
import os
DATA_DIR = '/home/tim-external/volumeROS/src/fsregistration/debug_results/3d'
# If you move this notebook, update DATA_DIR to point to the folder with the CSV files.

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider
import matplotlib
matplotlib.use('Agg')  # fallback only

def load_csv(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f'File not found: {filepath}')
        return None
    return np.loadtxt(filepath)

print(f'Data directory: {DATA_DIR}')
print(f'Rotation candidate: {which_rotation}')

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
# --- Build rotated filenames from config ---
if use_rotation_suffix:
    rot_suffix = str(which_rotation)
else:
    rot_suffix = ''

# --- Load voxel data (N x N x N) ---
magnitude_fftw1 = load_csv('magnitudeFFTW1.csv')
phase_fftw1 = load_csv('phaseFFTW1.csv')
voxel_data_used1 = load_csv('voxelDataFFTW1.csv')

magnitude_fftw2 = load_csv('magnitudeFFTW2.csv')
phase_fftw2 = load_csv('phaseFFTW2.csv')
voxel_data_used2 = load_csv('voxelDataFFTW2.csv')

magnitude_fftw2_rotated = load_csv(f'magnitudeFFTW2Rotated{rot_suffix}.csv')
phase_fftw2_rotated = load_csv(f'phaseFFTW2Rotated{rot_suffix}.csv')
voxel_data_used2_rotated = load_csv(f'voxelDataFFTW2Rotated{rot_suffix}.csv')

# --- Infer N and correlationN ---
if voxel_data_used1 is not None:
    N = int(round(len(voxel_data_used1) ** (1.0 / 3.0)))
    print(f'Inferred N = {N}')
else:
    raise ValueError('Cannot determine N: voxelDataFFTW1.csv not found')

correlationN = int(round(len(magnitude_fftw2_rotated) ** (1.0 / 3.0)))
print(f'Inferred correlationN = {correlationN}  (expected N*2-1 = {N*2-1})')

# --- Reshape 3D data (C++ writes i,j,k order = row-major [i][j][k]) ---
magnitude1 = magnitude_fftw1.reshape((N, N, N))
phase1 = phase_fftw1.reshape((N, N, N))
voxel_data1 = voxel_data_used1.reshape((N, N, N))
magnitude2 = magnitude_fftw2.reshape((N, N, N))
phase2 = phase_fftw2.reshape((N, N, N))
voxel_data2 = voxel_data_used2.reshape((N, N, N))
magnitude2_rotated = magnitude_fftw2_rotated.reshape((correlationN, correlationN, correlationN))
phase2_rotated = phase_fftw2_rotated.reshape((correlationN, correlationN, correlationN))
voxel_data2_rotated = voxel_data_used2_rotated.reshape((N, N, N))

# --- Resampled magnitude (N x N) ---
resampled_magnitude1 = load_csv('resampledMagnitudeSO3_1.csv')
resampled_magnitude2 = load_csv('resampledMagnitudeSO3_2.csv')
resampled_magnitude1_matrix = resampled_magnitude1.reshape((N, N))
resampled_magnitude2_matrix = resampled_magnitude2.reshape((N, N))

print(f'All data loaded. Voxel shape: ({N},{N},{N}), Correlation shape: ({correlationN},{correlationN},{correlationN})')

## Resampled Fourier Magnitudes

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Signal 1', 'Signal 2'))

fig.add_trace(
    go.Heatmap(z=resampled_magnitude1_matrix, colorscale='Viridis', colorshowscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=resampled_magnitude2_matrix, colorscale='Viridis'),
    row=1, col=2
)

fig.update_layout(height=450, width=900, title_text='Resampled Magnitude Comparison')
fig.update_xaxes(nticks=20)
fig.update_yaxes(nticks=20)
fig.show()

## Sphere Projection of Magnitude

In [ ]:
u = np.linspace(0, 2 * np.pi, N)
v = np.linspace(0, np.pi, N)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

fig = make_subplots(rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Sphere: Magnitude 1', 'Sphere: Magnitude 2'))

norm1 = resampled_magnitude1_matrix[:-1, :-1] / resampled_magnitude1_matrix.max()
norm2 = resampled_magnitude2_matrix[:-1, :-1] / resampled_magnitude2_matrix.max()

fig.add_trace(
    go.Surface(x=x_sphere, y=y_sphere, z=z_sphere,
               surfacecolor=norm1, colorscale='Viridis', showscale=False,
               contours_z=dict(show=True, usecolormap=True)),
    row=1, col=1
)
fig.add_trace(
    go.Surface(x=x_sphere, y=y_sphere, z=z_sphere,
               surfacecolor=norm2, colorscale='Viridis',
               contours_z=dict(show=True, usecolormap=True)),
    row=1, col=2
)

fig.update_layout(height=500, width=1000, title_text='Spherical Projection of Fourier Magnitude')
for ax in ['xaxis', 'yaxis', 'zaxis', 'xaxis2', 'yaxis2', 'zaxis2']:
    fig.update(**{ax: dict(nticks=6)})
fig.show()

## SO(3) Correlation Volume

In [ ]:
correlation_values_real = load_csv('resultingCorrelationReal.csv')
correlation_values_complex = load_csv('resultingCorrelationComplex.csv')

corr_side = int(round(len(correlation_values_real) ** (1.0 / 3.0)))
bw_out = corr_side // 2
print(f'Correlation grid: {corr_side}x{corr_side}x{corr_side}  (bwOut = {bw_out})')

correlation_values_real_matrix = correlation_values_real.reshape((corr_side, corr_side, corr_side))
correlation_values_complex_matrix = correlation_values_complex.reshape((corr_side, corr_side, corr_side))
correlation_values_magnitude = np.sqrt(
    correlation_values_real_matrix ** 2 + correlation_values_complex_matrix ** 2
)

print(f'Correlation magnitude — max: {correlation_values_magnitude.max():.6f}, min: {correlation_values_magnitude.min():.6f}')

@interact(slice_idx=(0, corr_side - 1, corr_side // 3))
def plot_correlation_slice(slice_idx=slice_idx):
    fig = make_subplots(rows=1, cols=2,
        specs=[[{'type': 'heatmap'}, {'type': 'scene'}]],
        subplot_titles=(f'XY Slice [{slice_idx}, :, :] (Heatmap)', f'XY Slice [{slice_idx}, :, :] (3D Surface)'))
    
    slice_data = correlation_values_magnitude[slice_idx, :, :]
    fig.add_trace(
        go.Heatmap(z=slice_data, colorscale='Hot', colorshowscale=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Surface(z=slice_data, colorscale='Hot', showscale=True,
                   colorbar_title='Magnitude'),
        row=1, col=2
    )
    fig.update_layout(height=500, width=1100,
        title_text=f'SO(3) Correlation Magnitude — XY Slice {slice_idx} / {corr_side-1} (rot {which_rotation})')
    fig.show()

## Translation Correlation Volume

In [ ]:
translation_correlation = load_csv(f'resultingCorrelationShift{rot_suffix}.csv')
corrN = int(round(len(translation_correlation) ** (1.0 / 3.0)))
print(f'Translation correlation grid: {corrN}x{corrN}x{corrN}')

translation_correlation_3d = translation_correlation.reshape((corrN, corrN, corrN))

# Peak detection
c_max = np.max(translation_correlation_3d.flatten())
idx_max = np.argmax(translation_correlation_3d.flatten())
peak_i, peak_j, peak_k = np.unravel_index(idx_max, translation_correlation_3d.shape)
center = corrN // 2
trans_x = peak_i - center
trans_y = peak_j - center
trans_z = peak_k - center

print(f'Peak index: ({peak_i}, {peak_j}, {peak_k})')
print(f'Max correlation: {c_max:.6f}')
print(f'Translation from center: ({trans_x}, {trans_y}, {trans_z})')

@interact(slice_idx=(0, corrN - 1, corrN // 2))
def plot_translation_slice(slice_idx=slice_idx):
    fig = make_subplots(rows=1, cols=2,
        specs=[[{'type': 'heatmap'}, {'type': 'scene'}]],
        subplot_titles=(f'XY Slice [{slice_idx}, :, :] (Heatmap)', f'XY Slice [{slice_idx}, :, :] (3D Surface)'))
    
    slice_data = translation_correlation_3d[slice_idx, :, :]
    fig.add_trace(
        go.Heatmap(z=slice_data, colorscale='Hot', colorshowscale=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Surface(z=slice_data, colorscale='Hot', showscale=True,
                   colorbar_title='Correlation'),
        row=1, col=2
    )
    fig.update_layout(height=500, width=1100,
        title_text=f'Translation Correlation — XY Slice {slice_idx} / {corrN-1} (rot {which_rotation})')
    fig.show()

## Peak Location Highlight

In [ ]:
# Show the slice containing the translation peak with a marker
fig = make_subplots(rows=1, cols=2,
    specs=[[{'type': 'heatmap'}, {'type': 'scene'}]],
    subplot_titles=(f'Peak Slice [X={peak_i}, :, :]', f'Peak Slice [X={peak_i}, :, :]'))

peak_slice = translation_correlation_3d[peak_i, :, :]

fig.add_trace(
    go.Heatmap(z=peak_slice, colorscale='Hot', colorshowscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Surface(z=peak_slice, colorscale='Hot', showscale=True),
    row=1, col=2
)

# Mark peak position on heatmap
fig.add_trace(
    go.Scatter(x=[peak_k], y=[peak_j], mode='markers',
               marker=dict(size=14, color='red', symbol='x', line_width=2),
               name='Peak'),
    row=1, col=1
)

fig.update_layout(height=500, width=1100,
    title_text=f'Translation Peak at index ({peak_i}, {peak_j}, {peak_k}) — value {c_max:.6f}')
fig.show()

print(f'Translation (voxel units from center): X={trans_x}, Y={trans_y}, Z={trans_z}')

## Compare Rotation Candidates

In [ ]:
# Quick comparison of translation peak heights across rotation candidates
import glob

shift_files = sorted(glob.glob(os.path.join(DATA_DIR, 'resultingCorrelationShift*.csv')))
print(f'Found {len(shift_files)} rotation candidates:\n')
print(f'{"Rot":>4}  {"Grid":>6}  {"Max Correlation":>16}  {"Peak Index":>20}  {"Translation (from center)":>30}')
print('-' * 80)

for fpath in shift_files:
    fname = os.path.basename(fpath)
    rot_id = fname.replace('resultingCorrelationShift', '').replace('.csv', '')
    data = np.loadtxt(fpath)
    sz = int(round(len(data) ** (1.0 / 3.0)))
    vol = data.reshape((sz, sz, sz))
    mx = np.max(vol)
    idx = np.unravel_index(np.argmax(vol), vol.shape)
    ctr = sz // 2
    tx, ty, tz = idx[0] - ctr, idx[1] - ctr, idx[2] - ctr
    print(f'{rot_id:>4}  {sz:>6}  {mx:>16.6f}  ({idx[0]:>3}, {idx[1]:>3}, {idx[2]:>3})  ({tx:>3}, {ty:>3}, {tz:>3})')